# 05. 합성 파형 대량 생성 (VAE → HiFi-GAN)

**목적**: finetune-m4 VAE + HiFi-GAN 파이프라인으로 M=4, 5, 6, 7 조건 합성 파형 생성

**실행 순서**
1. 셀 1~6 순서대로 실행 (환경 설정)
2. 셀 7: 생성 조건 설정 (M별 생성 개수 조정 가능)
3. 셀 8: 대량 생성 실행
4. 셀 9: 생성 결과 검증 (PGV 단조성 확인)

In [1]:
# 셀 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. GitHub 클론
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/seismic-gen'
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/isseul/cose362-k08-seismic-generation.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# HiFi-GAN 원본 클론 (utils, env 등 필요)
if not os.path.exists('/content/hifi-gan'):
    !git clone https://github.com/jik876/hifi-gan.git /content/hifi-gan

import sys
sys.path.insert(0, '/content/hifi-gan')
print('✅ 완료')

Cloning into '/content/seismic-gen'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 135 (delta 47), reused 101 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 6.01 MiB | 14.34 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Cloning into '/content/hifi-gan'...
remote: Enumerating objects: 48, done.
remote: Total 48 (delta 0), reused 0 (delta 0), pack-reused 48 (from 1)
Receiving objects: 100% (48/48), 620.94 KiB | 2.48 MiB/s, done.
Resolving deltas: 100% (20/20), done.
✅ 완료


In [3]:
# 셀 3. 라이브러리 & 경로 설정
!pip install torch torchaudio numpy pandas h5py -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os, json, glob
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# ── 경로 설정 ──────────────────────────────────────────
VAE_CKPT    = '/content/drive/MyDrive/ML_Project/outputs/film_vae/준우new-C1-beta1.0-spec1.0-mr0.5-rank5.0-encfilm-finetune-m4/film_vae_best.pth'
HIGAN_CKPT  = '/content/drive/MyDrive/ML_Project/outputs/hifigan_finetune_m4/hifigan_best.pth'
NORM_STATS  = '/content/drive/MyDrive/ML_Project/outputs/film_vae/준우new-C1-beta1.0-spec1.0-mr0.5-rank5.0-encfilm-finetune-m4/cond_norm_stats.json'
OUTPUT_DIR  = '/content/drive/MyDrive/ML_Project/outputs/synthetic_waveforms'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── STFT 설정 (학습 때와 동일) ─────────────────────────
SR         = 100
N_FFT      = 160
HOP_LENGTH = 46
WIN_LENGTH = 160
N_FREQ     = N_FFT // 2 + 1   # 81
N_SAMPLES  = 6000
N_TIME     = N_SAMPLES // HOP_LENGTH + 1  # 131

COND_COLS = [
    'source_magnitude',
    'source_latitude', 'source_longitude',
    'receiver_latitude', 'receiver_longitude',
    'source_depth_km'
]

LATENT_DIM = 16
HIDDEN_DIM = 256

print('✅ 설정 완료')

device: cuda
✅ 설정 완료


In [15]:
# 셀 4. VAE 모델 정의 (04_film_vae_finetune_m4.ipynb 와 완전히 동일)
class FiLMGenerator(nn.Module):
    def __init__(self, cond_dim=6, num_layers=3, hidden_dim=128, feature_dim=256):
        super().__init__()
        self.num_layers = num_layers
        self.shared_mlp = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.gamma_layers = nn.ModuleList([nn.Linear(hidden_dim, feature_dim) for _ in range(num_layers)])
        self.beta_layers  = nn.ModuleList([nn.Linear(hidden_dim, feature_dim) for _ in range(num_layers)])

    def forward(self, c):
        h = self.shared_mlp(c)
        return [g(h) for g in self.gamma_layers], [b(h) for b in self.beta_layers]


class FiLMLayer(nn.Module):
    def forward(self, h, gamma, beta):
        return gamma.unsqueeze(1) * h + beta.unsqueeze(1)


class FiLMEncoder(nn.Module):
    def __init__(self, in_freq=81, latent_dim=16, hidden=256, cond_dim=6):
        super().__init__()
        self.gru      = nn.GRU(in_freq, hidden, num_layers=2,
                               batch_first=True, bidirectional=True)
        self.film_gen = FiLMGenerator(cond_dim, num_layers=1,
                                      hidden_dim=128, feature_dim=hidden*2)
        self.film     = FiLMLayer()
        self.fc_mu    = nn.Linear(hidden * 2, latent_dim)
        self.fc_lv    = nn.Linear(hidden * 2, latent_dim)

    def forward(self, x, c):
      out, _ = self.gru(x)
      h = out[:, -1, :]   # 마지막 타임스텝
      gammas, betas = self.film_gen(c)
      h = self.film(h.unsqueeze(1), gammas[0], betas[0]).squeeze(1)
      return self.fc_mu(h), self.fc_lv(h)


class FiLMDecoder(nn.Module):
    def __init__(self, latent_dim=16, cond_dim=6, out_freq=81, n_time=131, hidden=256):
        super().__init__()
        self.n_time     = n_time
        self.num_layers = 3
        self.fc_z       = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU())
        self.gru_layers = nn.ModuleList([
            nn.GRU(hidden, hidden, batch_first=True) for _ in range(self.num_layers)
        ])
        self.film_gen    = FiLMGenerator(cond_dim, self.num_layers, 128, hidden)
        self.film_layers = nn.ModuleList([FiLMLayer() for _ in range(self.num_layers)])
        self.fc_out      = nn.Linear(hidden, out_freq)

    def forward(self, z, c):
        h = self.fc_z(z).unsqueeze(1).expand(-1, self.n_time, -1)
        gammas, betas = self.film_gen(c)
        for i in range(self.num_layers):
            h, _ = self.gru_layers[i](h)
            h    = self.film_layers[i](h, gammas[i], betas[i])
        return self.fc_out(h)


class FiLMVAE(nn.Module):
    def __init__(self, in_freq=81, n_time=131, latent_dim=16, cond_dim=6, hidden=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder    = FiLMEncoder(in_freq, latent_dim, hidden, cond_dim)
        self.decoder    = FiLMDecoder(latent_dim, cond_dim, in_freq, n_time, hidden)

    def reparameterize(self, mu, log_var):
        return mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)

    def forward(self, x, c):
        mu, log_var = self.encoder(x, c)
        z           = self.reparameterize(mu, log_var)
        return self.decoder(z, c), mu, log_var

    def generate(self, c):
        z = torch.randn(c.size(0), self.latent_dim, device=c.device)
        return self.decoder(z, c)

print('✅ VAE 모델 정의 완료')

✅ VAE 모델 정의 완료


In [16]:
# 셀 5. HiFi-GAN Generator 정의 (02_hifigan_train 과 동일)
from env import AttrDict
from utils import init_weights, get_padding
from torch.nn import Conv1d, ConvTranspose1d
from torch.nn.utils import weight_norm, remove_weight_norm

LRELU_SLOPE = 0.1

class ResBlock1(nn.Module):
    def __init__(self, h, channels, kernel_size=3, dilation=(1, 3, 5)):
        super().__init__()
        self.convs1 = nn.ModuleList([
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[0], padding=get_padding(kernel_size, dilation[0]))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[1], padding=get_padding(kernel_size, dilation[1]))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[2], padding=get_padding(kernel_size, dilation[2])))
        ])
        self.convs2 = nn.ModuleList([
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1)))
        ])
        self.convs1.apply(init_weights)
        self.convs2.apply(init_weights)

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = F.leaky_relu(x, LRELU_SLOPE)
            xt = c1(xt)
            xt = F.leaky_relu(xt, LRELU_SLOPE)
            xt = c2(xt)
            x  = xt + x
        return x


class Generator(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.h = h
        self.num_kernels   = len(h.resblock_kernel_sizes)
        self.num_upsamples = len(h.upsample_rates)
        self.conv_pre = weight_norm(Conv1d(h.num_mels, h.upsample_initial_channel, 7, 1, padding=3))
        self.ups = nn.ModuleList()
        for i, (u, k) in enumerate(zip(h.upsample_rates, h.upsample_kernel_sizes)):
            self.ups.append(weight_norm(ConvTranspose1d(
                h.upsample_initial_channel//(2**i),
                h.upsample_initial_channel//(2**(i+1)),
                k, u, padding=(k-u)//2
            )))
        self.resblocks = nn.ModuleList()
        for i in range(len(self.ups)):
            ch = h.upsample_initial_channel//(2**(i+1))
            for j, (k, d) in enumerate(zip(h.resblock_kernel_sizes, h.resblock_dilation_sizes)):
                self.resblocks.append(ResBlock1(h, ch, k, d))
        self.conv_post = weight_norm(Conv1d(ch, 1, 7, 1, padding=3))
        self.ups.apply(init_weights)
        self.conv_post.apply(init_weights)

    def forward(self, x):
        x = self.conv_pre(x)
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, LRELU_SLOPE)
            x = self.ups[i](x)
            xs = None
            for j in range(self.num_kernels):
                xs = self.resblocks[i*self.num_kernels+j](x) if xs is None else xs + self.resblocks[i*self.num_kernels+j](x)
            x = xs / self.num_kernels
        x = F.leaky_relu(x)
        x = self.conv_post(x)
        x = torch.tanh(x)
        if x.shape[-1] > N_SAMPLES:   x = x[..., :N_SAMPLES]
        elif x.shape[-1] < N_SAMPLES: x = F.pad(x, (0, N_SAMPLES - x.shape[-1]))
        return x

print('✅ HiFi-GAN Generator 정의 완료')

✅ HiFi-GAN Generator 정의 완료


In [17]:
# 셀 6. 모델 로드

# ── VAE 로드 ───────────────────────────────────────────
vae = FiLMVAE(
    in_freq=N_FREQ, n_time=N_TIME,
    latent_dim=LATENT_DIM, cond_dim=len(COND_COLS),
    hidden=HIDDEN_DIM
).to(device)

vae_ckpt = torch.load(VAE_CKPT, map_location=device)
vae.load_state_dict(vae_ckpt['model'])
vae.eval()
print(f'✅ VAE 로드 완료 (epoch={vae_ckpt["epoch"]}, val={vae_ckpt["val_loss"]:.4f})')

# WFS 정규화 통계 로드
with open(NORM_STATS) as f:
    stats = json.load(f)
cond_min = torch.tensor([stats['min'][c] for c in COND_COLS], dtype=torch.float32)
cond_max = torch.tensor([stats['max'][c] for c in COND_COLS], dtype=torch.float32)

# WFS_MIN, WFS_MAX는 체크포인트에 저장된 경우 로드

WFS_MIN = -10.0
WFS_MAX = 1.7679708003997803

print(f'WFS_MIN={WFS_MIN}, WFS_MAX={WFS_MAX}')

# ── HiFi-GAN 로드 ──────────────────────────────────────
higan_config = {
    "resblock": "1",
    "upsample_rates": [8, 8, 2, 2],
    "upsample_kernel_sizes": [16, 16, 4, 4],
    "upsample_initial_channel": 256,
    "resblock_kernel_sizes": [3, 7, 11],
    "resblock_dilation_sizes": [[1,3,5],[1,3,5],[1,3,5]],
    "num_mels": 81,
    "n_fft": 160, "hop_size": 46, "win_size": 160,
    "sampling_rate": 100, "segment_size": 6000
}
h = AttrDict(higan_config)
G = Generator(h).to(device)
higan_ckpt = torch.load(HIGAN_CKPT, map_location=device)
G.load_state_dict(higan_ckpt['G'])
G.eval()
print(f'✅ HiFi-GAN 로드 완료 (epoch={higan_ckpt["epoch"]}, G={higan_ckpt["G_loss"]:.4f})')

✅ VAE 로드 완료 (epoch=10, val=25.7986)
WFS_MIN=-10.0, WFS_MAX=1.7679708003997803
✅ HiFi-GAN 로드 완료 (epoch=73, G=86.3466)


In [18]:
# 셀 7. 생성 조건 설정
# ─────────────────────────────────────────────────────────────────────
# M별 생성 개수: EQTransformer 실험에서 실제 M≥4 데이터 (N_real) 기준
#   r=1  → 합성 N_real개    (1배)
#   r=10 → 합성 N_real*10개 (10배)
#   r=50 → 합성 N_real*50개 (50배)
#
# 우선 r=50 기준으로 최대량 생성해두고, 실험 시 슬라이싱해서 사용
# ─────────────────────────────────────────────────────────────────────

# 실제 M≥4 데이터 개수 (val set 기준 N=200을 test로 분리한다고 가정)
# → EQTransformer 실험 설계에서 확정 후 수정

N_REAL_PER_M = 200
R_MAX = 100

# M별 생성 개수 및 기본 조건값 (STEAD 통계 기반 대표값)
GENERATION_CONFIG = [
    {'M': 4.0, 'n': N_REAL_PER_M * R_MAX,
     'lat': 37.5, 'lon': -122.0, 'rec_lat': 37.6, 'rec_lon': -122.1, 'depth': 10.0},
    {'M': 5.0, 'n': N_REAL_PER_M * R_MAX,
     'lat': 37.5, 'lon': -122.0, 'rec_lat': 37.6, 'rec_lon': -122.1, 'depth': 10.0},
    {'M': 6.0, 'n': N_REAL_PER_M * R_MAX,
     'lat': 37.5, 'lon': -122.0, 'rec_lat': 37.6, 'rec_lon': -122.1, 'depth': 10.0},
    {'M': 7.0, 'n': N_REAL_PER_M * R_MAX,
     'lat': 37.5, 'lon': -122.0, 'rec_lat': 37.6, 'rec_lon': -122.1, 'depth': 10.0},
]

BATCH_SIZE = 32  # GPU 메모리에 따라 조정 (Colab T4: 32 권장)

print('생성 계획:')
total = 0
for cfg in GENERATION_CONFIG:
    print(f'  M={cfg["M"]} → {cfg["n"]:,}개')
    total += cfg['n']
print(f'총 {total:,}개 생성 예정')

생성 계획:
  M=4.0 → 20,000개
  M=5.0 → 20,000개
  M=6.0 → 20,000개
  M=7.0 → 20,000개
총 80,000개 생성 예정


In [19]:
# 셀 8. 대량 생성 실행
import time

def make_cond_tensor(M, lat, lon, rec_lat, rec_lon, depth, n, cond_min, cond_max):
    """조건 벡터 생성 및 정규화"""
    raw = torch.tensor([[M, lat, lon, rec_lat, rec_lon, depth]], dtype=torch.float32)
    raw = raw.expand(n, -1)  # (n, 6)
    cond_norm = (raw - cond_min) / (cond_max - cond_min + 1e-10)
    return cond_norm.clamp(0, 1)


@torch.no_grad()
def generate_batch(vae, G, cond_batch, wfs_min, wfs_max):
    """
    VAE → 스펙트로그램 → HiFi-GAN → 파형
    Returns: waveforms (np.array, shape: [B, 6000])
    """
    cond_batch = cond_batch.to(device)

    # 1. VAE: 조건 → 스펙트로그램 (log10 정규화 도메인)
    spec_norm = vae.generate(cond_batch)  # (B, N_TIME, N_FREQ), [0,1]

    # 2. 역정규화: [0,1] → log10 진폭
    spec_log = spec_norm * (wfs_max - wfs_min) + wfs_min  # (B, N_TIME, N_FREQ)

    # 3. HiFi-GAN 입력 형식: (B, N_FREQ, N_TIME)
    spec_input = spec_log.permute(0, 2, 1)  # (B, N_FREQ, N_TIME)

    # 4. HiFi-GAN: 스펙트로그램 → 파형
    wf = G(spec_input)  # (B, 1, 6000)
    wf = wf.squeeze(1).cpu().numpy()  # (B, 6000)

    return wf


# ── 생성 실행 ──────────────────────────────────────────
all_metadata = []  # EQTransformer 학습용 메타데이터

for cfg in GENERATION_CONFIG:
    M       = cfg['M']
    n_total = cfg['n']
    save_dir = os.path.join(OUTPUT_DIR, f'M{M:.1f}')
    os.makedirs(save_dir, exist_ok=True)

    print(f'\n[M={M}] {n_total:,}개 생성 시작...')
    t0 = time.time()
    generated = 0

    with tqdm(total=n_total, desc=f'M={M}') as pbar:
        while generated < n_total:
            bs = min(BATCH_SIZE, n_total - generated)
            cond = make_cond_tensor(
                M, cfg['lat'], cfg['lon'],
                cfg['rec_lat'], cfg['rec_lon'], cfg['depth'],
                bs, cond_min, cond_max
            )
            wfs = generate_batch(vae, G, cond, WFS_MIN, WFS_MAX)

            for i, wf in enumerate(wfs):
                idx = generated + i
                fname = f'syn_M{M:.1f}_{idx:06d}.npy'
                fpath = os.path.join(save_dir, fname)
                np.save(fpath, wf.astype(np.float32))

                all_metadata.append({
                    'file_path': fpath,
                    'source_magnitude': M,
                    'is_synthetic': True,
                    'split': 'train'  # EQTransformer 학습 시 train으로 사용
                })

            generated += bs
            pbar.update(bs)

    elapsed = time.time() - t0
    print(f'  완료: {n_total:,}개 / {elapsed:.1f}s ({elapsed/n_total*1000:.1f}ms/개)')

# 메타데이터 저장
meta_df = pd.DataFrame(all_metadata)
meta_df.to_csv(os.path.join(OUTPUT_DIR, 'synthetic_metadata.csv'), index=False)
print(f'\n✅ 전체 생성 완료: {len(meta_df):,}개')
print(meta_df.groupby('source_magnitude')['file_path'].count())


[M=4.0] 20,000개 생성 시작...


M=4.0: 100%|██████████| 20000/20000 [07:02<00:00, 47.33it/s]


  완료: 20,000개 / 422.5s (21.1ms/개)

[M=5.0] 20,000개 생성 시작...


M=5.0: 100%|██████████| 20000/20000 [07:53<00:00, 42.26it/s]


  완료: 20,000개 / 473.3s (23.7ms/개)

[M=6.0] 20,000개 생성 시작...


M=6.0: 100%|██████████| 20000/20000 [08:32<00:00, 39.05it/s]


  완료: 20,000개 / 512.1s (25.6ms/개)

[M=7.0] 20,000개 생성 시작...


M=7.0: 100%|██████████| 20000/20000 [08:37<00:00, 38.65it/s]


  완료: 20,000개 / 517.4s (25.9ms/개)

✅ 전체 생성 완료: 80,000개
source_magnitude
4.0    20000
5.0    20000
6.0    20000
7.0    20000
Name: file_path, dtype: int64


In [22]:
# 셀 9. 생성 결과 검증 (PGV 단조성 + 파형 시각화)
import matplotlib.pyplot as plt

N_VERIFY = 20  # M당 검증 샘플 수
M_targets = [4.0, 5.0, 6.0, 7.0]
pgv_list = []

fig, axes = plt.subplots(len(M_targets), 1, figsize=(14, 10))

# VAE 스펙트로그램 도메인에서 PGV 계산 (04 노트북 방식)
for i, M in enumerate(M_targets):
    cvals = [M, 37.5, -122.0, 37.6, -122.1, 10.0]
    ct = torch.tensor(cvals, dtype=torch.float32).unsqueeze(0).to(device)
    ct = ((ct - cond_min.to(device)) / (cond_max.to(device) - cond_min.to(device) + 1e-10)).clamp(0, 1)

    pgv_samples = []
    with torch.no_grad():
        for _ in range(20):
            spec = vae.generate(ct)[0].cpu()  # (N_TIME, N_FREQ)
            X_denorm = spec * (WFS_MAX - WFS_MIN) + WFS_MIN
            mag = torch.pow(10, X_denorm).clamp(1e-10)
            pgv_samples.append(mag.mean().item())  # 스펙트로그램 평균

    pgv_list.append(np.mean(pgv_samples))

    # 첫 번째 파형 시각화
    wf_sample = np.load(files[0])
    axes[i].plot(wf_sample, lw=0.5, color='steelblue', alpha=0.8)
    axes[i].set_title(f'M={M} | PGV(mean)={mean_pgv:.5f} ± {np.std(pgvs):.5f}', fontsize=11)
    axes[i].set_ylabel('Amplitude')
    axes[i].axhline(0, color='k', lw=0.3)

axes[-1].set_xlabel('Sample (100Hz)')
plt.suptitle('합성 파형 검증 (VAE → HiFi-GAN)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'generated_verification.png'), dpi=150)
plt.show()

print('\n── PGV 단조성 검증 ──')
for M, pgv in zip(M_targets, pgv_list):
    print(f'  M={M} → PGV={pgv:.5f}')

is_monotone = all(pgv_list[i] <= pgv_list[i+1] for i in range(len(pgv_list)-1))
print('\n✅ PGV 단조 증가 확인' if is_monotone else '\n⚠️  단조 증가 실패 — 생성 조건 확인 필요')

/tmp/ipykernel_675/4175869880.py:35: UserWarning: Glyph 54633 (\N{HANGUL SYLLABLE HAB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_675/4175869880.py:35: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_675/4175869880.py:35: UserWarning: Glyph 54028 (\N{HANGUL SYLLABLE PA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_675/4175869880.py:35: UserWarning: Glyph 54805 (\N{HANGUL SYLLABLE HYEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_675/4175869880.py:35: UserWarning: Glyph 44160 (\N{HANGUL SYLLABLE GEOM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_675/4175869880.py:35: UserWarning: Glyph 51613 (\N{HANGUL SYLLABLE JEUNG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_675/4175869880.py:36: UserWarning: Glyph 54633 (\N{HANGUL SYLLABLE HAB}) missing from font(s) DejaVu Sans.
  plt.savefig


── PGV 단조성 검증 ──
  M=4.0 → PGV=0.22997
  M=5.0 → PGV=0.37478
  M=6.0 → PGV=0.76088
  M=7.0 → PGV=1.33870

✅ PGV 단조 증가 확인


In [24]:
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/seismic-gen'
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/isseul/cose362-k08-seismic-generation.git'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.system(f'cd {REPO_DIR} && git fetch origin')
os.system(f'cd {REPO_DIR} && git checkout vae-dev')

NOTEBOOK_SRC = '/content/drive/MyDrive/05_generate_synthetic_waveforms.ipynb'
NOTEBOOK_DST = f'{REPO_DIR}/vae/05_generate_synthetic_waveforms.ipynb'

os.makedirs(f'{REPO_DIR}/vae', exist_ok=True)
os.system(f'cp "{NOTEBOOK_SRC}" "{NOTEBOOK_DST}"')

os.system(f'cd {REPO_DIR} && git config user.email "joonoo.kwak@gmail.com"')
os.system(f'cd {REPO_DIR} && git config user.name "Team-K8"')
os.system(f'cd {REPO_DIR} && git add vae/05_generate_synthetic_waveforms.ipynb')
os.system(f'cd {REPO_DIR} && git commit -m "feat(vae): 05 합성 파형 대량 생성 완료 (M=4~7, 80000개)"')
os.system(f'cd {REPO_DIR} && git push origin vae-dev')
print('✅ push 완료')

✅ push 완료
